In [1]:


import numpy as np
import pandas as pd

# ============================================================
# STEP 1: Load Original Dataset
# ============================================================

INPUT_FILE = (
    "/content/drive/MyDrive/My work/Air Quality Project/Paper 1(External Data)/New/Merge.csv"
)

OUTPUT_FILE = "merged_aqi_corrected.csv"

df = pd.read_csv(INPUT_FILE, low_memory=False)

print("Original Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())


# ============================================================
# STEP 2: Define Pollutants
# ============================================================

pollutant_cols = [
    'SO2', 'NO2', 'CO',
    'O3', 'PM2.5', 'PM10'
]


breakpoints = {
    'PM2.5': [
        (0, 30),
        (31, 60),
        (61, 90),
        (91, 120),
        (121, 250)
    ],

    'PM10': [
        (0, 50),
        (51, 150),
        (151, 250),
        (251, 350),
        (351, 430)
    ],

    'NO2': [
        (0, 40),
        (41, 80),
        (81, 180),
        (181, 280),
        (281, 400)
    ],

    'O3': [
        (0, 50),
        (51, 100),
        (101, 168),
        (169, 208),
        (209, 748)
    ],

    'CO': [
        (0, 1.0),
        (1.1, 2.0),
        (2.1, 10.0),
        (10.1, 17.0),
        (17.1, 34.0)
    ],

    'SO2': [
        (0, 40),
        (41, 80),
        (81, 380),
        (381, 800),
        (801, 1600)
    ]
}

# AQI ranges for the first five categories
aqi_ranges = [
    (0, 50),
    (51, 100),
    (101, 200),
    (201, 300),
    (301, 400)
]

categories = [
    'Good',
    'Satisfactory',
    'Moderate',
    'Poor',
    'Very Poor',
    'Severe'
]



def calculate_sub_index_bd(pollutant, concentration):

    if pollutant not in breakpoints:
        raise ValueError(
            f"Unknown pollutant: {pollutant}"
        )

    if pd.isna(concentration):
        return np.nan, None

    concentration = float(concentration)

    if not np.isfinite(concentration):
        return np.nan, None

    if concentration < 0:
        return np.nan, None

    pollutant_ranges = breakpoints[pollutant]

    upper_bounds = [
        item[1] for item in pollutant_ranges
    ]

    # Assign continuous concentrations using upper limits.
    # For example:
    # 30.0 -> Good
    # 30.5 -> Satisfactory
    # 31.0 -> Satisfactory

    category_code = int(
        np.searchsorted(
            upper_bounds,
            concentration,
            side='left'
        )
    )


    if category_code == 5:
        return np.nan, 5

    # Corresponding concentration range
    C_low, C_high = pollutant_ranges[category_code]

    # Corresponding AQI range
    I_low, I_high = aqi_ranges[category_code]

    # Linear interpolation
    sub_index = (
        ((I_high - I_low) / (C_high - C_low))
        * (concentration - C_low)
        + I_low
    )

    sub_index = np.clip(
        sub_index,
        I_low,
        I_high
    )

    return int(round(sub_index)), category_code


# ============================================================
# STEP 5: Convert Pollutant Columns to Numeric
# ============================================================

aqi_df = df.copy()

for pollutant in pollutant_cols:

    aqi_df[pollutant] = pd.to_numeric(
        aqi_df[pollutant],
        errors='coerce'
    )


values = aqi_df[pollutant_cols].to_numpy(dtype=float)

valid_rows = (
    np.isfinite(values).all(axis=1)
    & (values >= 0).all(axis=1)
)

print("\n=== Data Quality ===")
print("Total rows:", len(aqi_df))
print("Valid rows:", valid_rows.sum())
print("Invalid rows:", (~valid_rows).sum())


# ============================================================
# STEP 6: Calculate Subindices and Pollutant Categories
# ============================================================

for pollutant in pollutant_cols:

    calculated = aqi_df[pollutant].apply(
        lambda x: calculate_sub_index_bd(
            pollutant, x
        )
    )

    aqi_df[f'{pollutant}_AQI'] = calculated.apply(
        lambda result: result[0]
    )

    aqi_df[f'{pollutant}_ClassCode'] = calculated.apply(
        lambda result: result[1]
    )


# ============================================================
# STEP 7: Calculate Overall AQI Category
# Based on highest pollutant category
# ============================================================

class_columns = [
    f'{p}_ClassCode' for p in pollutant_cols
]

subindex_columns = [
    f'{p}_AQI' for p in pollutant_cols
]

# Each invalid row remains unclassified.
class_matrix = aqi_df[class_columns].to_numpy(
    dtype=float
)

class_matrix = np.where(
    np.isfinite(class_matrix),
    class_matrix,
    -1
)

overall_codes = class_matrix.max(axis=1).astype(int)

overall_codes[~valid_rows] = -1

category_output = np.full(
    len(aqi_df),
    None,
    dtype=object
)

category_output[valid_rows] = np.array(
    categories,
    dtype=object
)[overall_codes[valid_rows]]

aqi_df['AQI_Category'] = pd.Series(
    category_output,
    index=aqi_df.index,
    dtype='string'
)


# ============================================================
# STEP 8: Calculate Numerical AQI
# ============================================================

subindex_matrix = aqi_df[
    subindex_columns
].to_numpy(dtype=float)

safe_subindices = np.where(
    np.isfinite(subindex_matrix),
    subindex_matrix,
    -np.inf
)

maximum_subindex = safe_subindices.max(axis=1)

numeric_valid = valid_rows & (overall_codes < 5)

aqi_values = np.where(
    numeric_valid,
    maximum_subindex,
    np.nan
)

aqi_df['AQI'] = pd.Series(
    aqi_values,
    index=aqi_df.index
).astype('Int64')


aqi_df['Is_Severe'] = (
    aqi_df['AQI_Category'] == 'Severe'
).fillna(False)


aqi_df['AQI_Severe_LowerBound'] = np.where(
    aqi_df['Is_Severe'],
    401,
    np.nan
)

# Label quality status
aqi_df['AQI_Status'] = np.select(
    [
        ~valid_rows,
        overall_codes == 5
    ],
    [
        'Invalid measurement',
        'Severe - exact AQI unavailable'
    ],
    default='Calculated from manuscript thresholds'
)


# ============================================================
# STEP 9: Confirm Corrected Category Distribution
# ============================================================

print("\n=== Corrected AQI Distribution ===")

print(
    aqi_df['AQI_Category'].value_counts(
        dropna=False
    )
)

print("\n=== AQI Calculation Status ===")

print(
    aqi_df['AQI_Status'].value_counts()
)


# ============================================================
# STEP 10: Identify Pollutants Triggering Severe
# ============================================================

print("\n=== Severe Trigger Counts ===")

severe_triggers = {}

for pollutant in pollutant_cols:

    maximum_bound = breakpoints[pollutant][-1][1]

    severe_triggers[pollutant] = int(
        (
            aqi_df[pollutant] > maximum_bound
        ).sum()
    )

trigger_df = pd.DataFrame({
    'Pollutant': list(severe_triggers.keys()),
    'Severe Triggers': list(severe_triggers.values())
})

print(
    trigger_df.sort_values(
        'Severe Triggers',
        ascending=False
    ).to_string(index=False)
)

# ============================================================
# STEP 11: Check Dataset Consistency
# ============================================================

print("\n=== Final Quality Check ===")

print("Total records:", len(aqi_df))

print(
    "Classified records:",
    aqi_df['AQI_Category'].notna().sum()
)

print(
    "Severe records:",
    aqi_df['Is_Severe'].sum()
)

print(
    "Records with numerical AQI:",
    aqi_df['AQI'].notna().sum()
)

print(
    "Invalid records:",
    (~valid_rows).sum()
)


# ============================================================
# STEP 12: Save Corrected Dataset and Audit
# ============================================================

aqi_df.to_csv(
    OUTPUT_FILE,
    index=False
)

trigger_df.to_csv(
    'AQI_Severe_Trigger_Summary.csv',
    index=False
)

print("\nCorrected dataset saved:", OUTPUT_FILE)
print("Severe trigger summary saved.")

# Preview
print("\n=== Dataset Preview ===")

print(
    aqi_df[
        [
            'PM2.5', 'PM10', 'NO2',
            'O3', 'CO', 'SO2',
            'AQI', 'AQI_Category',
            'AQI_Status'
        ]
    ].head(10).to_string(index=False)
)


Original Dataset Shape: (271334, 11)
Columns: ['Date', 'Time', 'CO', 'O3', 'PM2.5', 'PM10', 'SO2', 'NO2', 'Location', 'AQI', 'AQI_Category']

=== Data Quality ===
Total rows: 271334
Valid rows: 271326
Invalid rows: 8

=== Corrected AQI Distribution ===
AQI_Category
Moderate        170940
Satisfactory     37263
Very Poor        28081
Poor             14400
Good             11800
Severe            8842
<NA>                 8
Name: count, dtype: Int64

=== AQI Calculation Status ===
AQI_Status
Calculated from manuscript thresholds    262484
Severe - exact AQI unavailable             8842
Invalid measurement                           8
Name: count, dtype: int64

=== Severe Trigger Counts ===
Pollutant  Severe Triggers
    PM2.5             6887
     PM10             5845
       CO               83
      SO2                0
      NO2                0
       O3                0

=== Final Quality Check ===
Total records: 271334
Classified records: 271326
Severe records: 8842
Records with nu

In [3]:
# Ensure the cleaned DataFrame is defined in active memory first
aqi_df_cleaned = aqi_df[valid_rows].copy()

# Find how many null values exist in each column
null_counts = aqi_df_cleaned.isnull().sum()
print(null_counts)

Date                       8195
Time                          0
CO                            0
O3                            0
PM2.5                         0
PM10                          0
SO2                           0
NO2                           0
Location                      0
AQI                        8842
AQI_Category                  0
SO2_AQI                       0
SO2_ClassCode                 0
NO2_AQI                       0
NO2_ClassCode                 0
CO_AQI                       83
CO_ClassCode                  0
O3_AQI                        0
O3_ClassCode                  0
PM2.5_AQI                  6887
PM2.5_ClassCode               0
PM10_AQI                   5845
PM10_ClassCode                0
Is_Severe                     0
AQI_Severe_LowerBound    262484
AQI_Status                    0
dtype: int64


In [4]:
# Verify that all rows with null AQI in the cleaned dataset correspond to the 'Severe' category
null_aqi_data = aqi_df_cleaned[aqi_df_cleaned['AQI'].isna()]

print(f"Total rows with null AQI: {len(null_aqi_data)}")
print("\nDistribution of AQI_Category for these null AQI rows:")
print(null_aqi_data['AQI_Category'].value_counts(dropna=False))

Total rows with null AQI: 8842

Distribution of AQI_Category for these null AQI rows:
AQI_Category
Severe    8842
Name: count, dtype: Int64


In [ ]:
invalid_records = aqi_df[~valid_rows]
invalid_records

In [14]:
# Show the invalid records, focusing on the columns with negative values (like SO2)
invalid_records = aqi_df[~valid_rows]
invalid_records[['Date', 'Time', 'Location', 'SO2', 'PM2.5', 'PM10', 'CO', 'NO2', 'O3']]

,Date,Time,Location,SO2,PM2.5,PM10,CO,NO2,O3
34066,12/7/2020,17:00,Darussalam,-0.03,216.76,281.99,1.72,15.69,2.99
34069,12/7/2020,20:00,Darussalam,-0.02,176.87,209.65,1.67,10.73,2.57
34070,12/7/2020,21:00,Darussalam,-0.17,152.19,173.18,1.59,11.60,2.61
34073,12/7/2020,24:00:00,Darussalam,-0.09,118.08,138.81,1.42,8.97,2.43
34074,12/8/2020,1:00,Darussalam,-0.34,108.18,131.20,1.29,9.16,2.61
34075,12/8/2020,2:00,Darussalam,-0.11,103.02,121.53,1.22,8.64,2.54
34076,12/8/2020,3:00,Darussalam,-0.03,102.68,115.23,1.18,7.67,2.73
34077,12/8/2020,4:00,Darussalam,-0.01,98.33,123.04,1.16,7.59,2.83


In [7]:
# Drop the invalid rows from the DataFrame
aqi_df_cleaned = aqi_df[valid_rows].copy()
print("Original shape:", aqi_df.shape)
print("New shape after dropping invalid rows:", aqi_df_cleaned.shape)

Original shape: (271334, 26)
New shape after dropping invalid rows: (271326, 26)


In [8]:
columns_to_keep = ['Date', 'Time', 'Location', 'PM2.5', 'PM10', 'NO2', 'O3', 'CO', 'SO2', 'AQI', 'AQI_Category']
aqi_df_cleaned = aqi_df_cleaned[columns_to_keep]
aqi_df_cleaned.head()

,Date,Time,Location,PM2.5,PM10,NO2,O3,CO,SO2,AQI,AQI_Category
0,11/5/2012,14:00,Sangsad,16.13,27.09,4.90,3.29,0.47,10.99,27,Good
1,11/5/2012,15:00,Sangsad,19.90,30.45,4.91,4.58,0.45,10.89,33,Good
2,11/5/2012,16:00,Sangsad,17.10,43.94,4.80,4.96,0.45,11.28,44,Good
3,11/5/2012,18:00,Sangsad,15.58,41.43,4.51,3.06,0.57,11.83,41,Good
4,11/5/2012,19:00,Sangsad,32.94,54.17,4.51,2.96,0.56,12.00,54,Satisfactory


In [10]:
# Filter and display records where AQI is NaN (null)
missing_aqi_records = aqi_df_cleaned[aqi_df_cleaned['AQI'].isna()]
print(f"Number of records where AQI is NaN: {len(missing_aqi_records)}")
missing_aqi_records.head()

Number of records where AQI is NaN: 8842


,Date,Time,Location,PM2.5,PM10,NO2,O3,CO,SO2,AQI,AQI_Category
357,12/6/2012,4:00,Sangsad,308.20,415.40,6.94,3.46,3.49,14.90,<NA>,Severe
358,12/6/2012,8:00,Sangsad,302.15,361.44,5.92,3.53,3.54,14.06,<NA>,Severe
360,12/7/2012,1:00,Sangsad,263.63,343.19,5.74,3.31,3.44,14.50,<NA>,Severe
361,12/7/2012,3:00,Sangsad,304.69,395.26,5.26,3.35,3.42,13.62,<NA>,Severe
362,12/7/2012,4:00,Sangsad,295.59,367.63,3.48,3.37,3.34,12.53,<NA>,Severe


In [11]:
# Show which pollutants trigger the 'Severe' classification
trigger_summary = trigger_df.sort_values(by='Severe Triggers', ascending=False)
print("=== Pollutants Triggering Severe Air Quality ===")
print(trigger_summary.to_string(index=False))

# Calculate percent contribution of each component to overall severe triggers
total_triggers = trigger_summary['Severe Triggers'].sum()
trigger_summary['Percentage (%)'] = (trigger_summary['Severe Triggers'] / total_triggers * 100).round(2)
print("\n=== Percentage Contribution ===")
print(trigger_summary[['Pollutant', 'Percentage (%)']].to_string(index=False))

=== Pollutants Triggering Severe Air Quality ===
Pollutant  Severe Triggers
    PM2.5             6887
     PM10             5845
       CO               83
      SO2                0
      NO2                0
       O3                0

=== Percentage Contribution ===
Pollutant  Percentage (%)
    PM2.5           53.74
     PM10           45.61
       CO            0.65
      SO2            0.00
      NO2            0.00
       O3            0.00


In [12]:
# Show the upper concentration limit beyond which AQI cannot be calculated (Severe class)
print("=== Pollutant Upper Concentration Limits (Beyond which AQI becomes NaN) ===")
for pollutant, ranges in breakpoints.items():
    # The last tuple in breakpoints represents the Very Poor category range.
    # Any concentration above the upper bound of this category triggers 'Severe' and becomes NaN.
    upper_limit = ranges[-1][1]
    triggers = severe_triggers[pollutant]
    print(f"{pollutant:<6} : Concentration > {upper_limit:<5} | Triggers: {triggers}")

=== Pollutant Upper Concentration Limits (Beyond which AQI becomes NaN) ===
PM2.5  : Concentration > 250   | Triggers: 6887
PM10   : Concentration > 430   | Triggers: 5845
NO2    : Concentration > 400   | Triggers: 0
O3     : Concentration > 748   | Triggers: 0
CO     : Concentration > 34.0  | Triggers: 83
SO2    : Concentration > 1600  | Triggers: 0


In [13]:
cleaned_output_file = 'merged_aqi_corrected_cleaned.csv'
aqi_df_cleaned.to_csv(cleaned_output_file, index=False)
print(f"Cleaned dataset successfully saved to: {cleaned_output_file}")

Cleaned dataset successfully saved to: merged_aqi_corrected_cleaned.csv
